# Tutorial: LangChain en Python

Este cuaderno introduce LangChain usando el entorno `AIAerospace` y la API de OpenAI.

LangChain es una libreria para construir aplicaciones con modelos de lenguaje mediante componentes reutilizables: modelos, prompts, parsers, herramientas, memoria, recuperacion documental y cadenas.

Contenidos:

- Configuracion de modelos de chat.
- Prompts y LCEL (*LangChain Expression Language*).
- Salida estructurada.
- Tool calling.
- Memoria conversacional.
- RAG sencillo con `BOE-A-2015-2870.pdf`.

Referencias oficiales: [models](https://docs.langchain.com/oss/python/langchain/models), [messages](https://docs.langchain.com/oss/python/langchain/messages), [retrieval](https://docs.langchain.com/oss/python/langchain/retrieval), [RAG tutorial](https://docs.langchain.com/oss/python/langchain/rag) y [OpenAIEmbeddings](https://docs.langchain.com/oss/python/integrations/embeddings/openai).

## 1. Preparacion

Antes de ejecutar el cuaderno, activa el entorno `AIAerospace` y define `OPENAI_API_KEY`.

```powershell
conda activate AIAerospace
$env:OPENAI_API_KEY="sk-..."
jupyter lab
```

El modelo se centraliza en `MODEL` para poder cambiarlo sin modificar todas las celdas.

In [2]:
import os
import textwrap
from pathlib import Path

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

def show(text, width=95):
    print(textwrap.fill(str(text), width=width))

print("Modelo de chat:", MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)

Modelo de chat: gpt-5.4-nano
Modelo de embeddings: text-embedding-3-small


## 2. Invocar un modelo de chat

`init_chat_model` permite crear un modelo indicando proveedor y nombre. El resultado de `invoke` es normalmente un `AIMessage`; el texto esta en `.content`.

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

llm = init_chat_model(MODEL, model_provider="openai", temperature=0.2)

messages = [
    SystemMessage(content="Eres un profesor de IA aplicada a aeroespacio. Responde en espanol."),
    HumanMessage(content="Explica en 3 frases que problema resuelve LangChain."),
]

response = llm.invoke(messages)
show(response.content)

c:\Users\igome\miniconda3\envs\AIAerospace\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangChain resuelve principalmente el problema de **conectar modelos de lenguaje (LLMs) con
herramientas y fuentes de datos** para que las respuestas no sean solo “texto suelto”, sino
acciones y consultas útiles. También ayuda a **orquestar flujos complejos** (por ejemplo:
planificar, consultar documentación/KB, resumir y redactar) de forma modular y mantenible.
Además, facilita **la recuperación y uso de contexto** (p. ej., mediante RAG con bases de
conocimiento) para reducir alucinaciones y mejorar la calidad cuando se trabaja con información
externa.


## 3. Prompts y LCEL

LCEL permite encadenar componentes con `|`. Un patron habitual es:

`prompt | modelo | parser`

El parser `StrOutputParser` transforma el mensaje del modelo en texto plano.

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un docente exigente pero claro. Responde en {idioma}."),
    ("human", "Resume el siguiente concepto en {n_frases} frases: {concepto}"),
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "idioma": "espanol",
    "n_frases": 4,
    "concepto": "LCEL en LangChain",
})

show(answer)

LCEL (LangChain Expression Language) es el lenguaje para expresar **flujos y encadenamientos de
pasos** en LangChain de forma declarativa.   Permite **conectar componentes** (modelos,
prompts, recuperadores, herramientas y salidas) mediante operadores y composición.   Su enfoque
facilita construir **pipelines legibles, reutilizables y mantenibles**, incluyendo ramificación
y transformación de datos.   Además, ayuda a integrar mejor **RAG y agentes** porque
estandariza cómo se orquesta la lógica entre los distintos componentes.


### Batch y streaming

Los componentes de LangChain implementan una interfaz comun. La misma cadena puede usarse con `.invoke`, `.batch` o `.stream`.

In [5]:
inputs = [
    {"idioma": "espanol", "n_frases": 2, "concepto": "embeddings"},
    {"idioma": "espanol", "n_frases": 2, "concepto": "RAG"},
    {"idioma": "espanol", "n_frases": 2, "concepto": "tool calling"},
]

for concept, result in zip([item["concepto"] for item in inputs], chain.batch(inputs)):
    print("\nConcepto:", concept)
    show(result)


Concepto: embeddings
Los **embeddings** son representaciones numéricas (vectores) que describen palabras, frases o
imágenes en un espacio donde las similitudes semánticas se reflejan como cercanía entre
vectores. Se aprenden a partir de datos para que el modelo pueda medir relaciones y realizar
tareas como búsqueda, clasificación o recomendación usando distancias o operaciones
matemáticas.

Concepto: RAG
RAG (Retrieval-Augmented Generation) es un enfoque que combina búsqueda de información con
generación de texto: primero recupera datos relevantes de una base (por ejemplo, documentos) y
luego los usa para generar la respuesta. Así, mejora la calidad y la pertinencia del contenido
al basarse en fuentes externas en lugar de depender solo de conocimiento previo del modelo.

Concepto: tool calling
El **tool calling** es una función que permite que un modelo consulte herramientas externas
(por ejemplo, calculadoras, bases de datos o APIs) para obtener información o realizar
acciones. Así, 

## 4. Salida estructurada

Para integrar un LLM en una aplicacion, a menudo necesitamos objetos con campos validados en vez de texto libre. LangChain permite usar esquemas de Pydantic con `with_structured_output`.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

class IncidentClassification(BaseModel):
    system: Literal["motor", "pitot-estatico", "frenos", "estructura", "otro"] = Field(
        description="Sistema afectado por la incidencia."
    )
    severity: Literal["baja", "media", "alta", "desconocida"] = Field(
        description="Severidad estimada solo a partir del texto."
    )
    needs_human_review: bool = Field(description="Si requiere revision humana.")
    rationale: str = Field(description="Justificacion breve.")

structured_llm = llm.with_structured_output(IncidentClassification)

incident = "Tras el aterrizaje se detecta temperatura anomala en frenos y olor a material caliente."
classification = structured_llm.invoke(
    "Clasifica esta incidencia aeronautica. No inventes informacion no presente: " + incident
)

classification

IncidentClassification(system='frenos', severity='media', needs_human_review=True, rationale='Tras el aterrizaje se detecta temperatura anómala en frenos y olor a material caliente, lo que sugiere un sobrecalentamiento de frenos; requiere revisión humana por posible daño/incendio.')

## 5. Tool calling

Las herramientas conectan el modelo con funciones controladas por la aplicacion. En este ejemplo el modelo puede consultar una base de datos ficticia de aeronaves de entrenamiento.

In [7]:
from langchain_core.tools import tool

AIRCRAFT_DB = {
    "EC-LLM": {"type": "Cessna 172", "hours": 1830, "last_inspection_days": 21, "open_findings": 0},
    "EC-RAG": {"type": "Piper PA-28", "hours": 2415, "last_inspection_days": 67, "open_findings": 2},
    "EC-GPT": {"type": "Diamond DA40", "hours": 920, "last_inspection_days": 12, "open_findings": 1},
}

@tool
def get_aircraft_status(registration: str) -> dict:
    """Consulta el estado de una aeronave de entrenamiento por matricula."""
    registration = registration.upper().strip()
    aircraft = AIRCRAFT_DB.get(registration)
    if aircraft is None:
        return {"found": False, "registration": registration}
    return {"found": True, "registration": registration, **aircraft}

tools = [get_aircraft_status]
llm_with_tools = llm.bind_tools(tools)

tool_request = llm_with_tools.invoke(
    "Consulta EC-RAG y dime si hay algo que revisar antes de una practica docente."
)

print("Contenido del mensaje:", tool_request.content)
print("Llamadas a herramientas:")
tool_request.tool_calls

Contenido del mensaje: 
Llamadas a herramientas:


[{'name': 'get_aircraft_status',
  'args': {'registration': 'EC-RAG'},
  'id': 'call_HOoaCR9vpnwVA8z9urV8HL0F',
  'type': 'tool_call'}]

El modelo no ejecuta la funcion por si solo en este ejemplo. Devuelve una solicitud de herramienta y nuestro codigo decide que ejecutar. Despues pasamos el resultado al modelo para redactar la respuesta final.

In [8]:
from langchain_core.messages import ToolMessage

tool_messages = []

for call in tool_request.tool_calls:
    if call["name"] == "get_aircraft_status":
        result = get_aircraft_status.invoke(call["args"])
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke([
    HumanMessage(content="Consulta EC-RAG y dime si hay algo que revisar antes de una practica docente."),
    tool_request,
    *tool_messages,
])

show(final.content)

He consultado **EC-RAG** (Piper PA-28). Datos clave:  - **Última inspección:** hace **67 días**
- **Estado:** con **2 hallazgos abiertos** - **Horas:** **2415**  ✅ **Qué revisar antes de una
práctica docente (recomendado):** 1. **Los 2 hallazgos abiertos**: confirmar cuáles son, su
criticidad y si hay alguna restricción/limitación operativa asociada. 2. **Vigencia de la
inspección**: aunque no está “muy” vencida (67 días), revisar el calendario para asegurar que
no cae cerca/antes de tu fecha de vuelo. 3. **Cualquier acción correctiva pendiente** ligada a
esos hallazgos (documentación, soporte técnico y conformidad para operar).  Si me dices **qué
fecha/hora** es la práctica y si hay **detalles de esos 2 hallazgos** (o te interesa que los
revise con más contexto), te preparo una checklist específica para el briefing previo.


## 6. Memoria conversacional

`RunnableWithMessageHistory` permite envolver una cadena para que recuerde mensajes por `session_id`. En este cuaderno evitamos un bucle interactivo y simulamos tres turnos.

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un tutor de LangChain. Responde breve y recuerda datos dados por el usuario."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

memory_chain = memory_prompt | llm | StrOutputParser()
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat_with_memory = RunnableWithMessageHistory(
    memory_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "alumno-1"}}

turns = [
    "Me llamo Ignacio y estoy preparando una clase de IA aeroespacial.",
    "Explica LCEL en una frase.",
    "Como me llamo y que estoy preparando?",
]

for user_text in turns:
    print("\nUsuario:", user_text)
    assistant_text = chat_with_memory.invoke({"input": user_text}, config=config)
    print("Asistente:")
    show(assistant_text)


Usuario: Me llamo Ignacio y estoy preparando una clase de IA aeroespacial.
Asistente:
¡Perfecto, Ignacio! 👋 Como tutor de LangChain, puedo ayudarte a montar una clase de IA con un
enfoque aeroespacial (p. ej., simulación, navegación, estimación de estado, análisis de datos
de sensores, misiones, etc.).  Para ajustarlo a tu objetivo:   1) ¿Tu clase es para nivel
**universidad**, **empresa** o **secundaria**?   2) ¿Quieres usar LangChain para **RAG**
(documentos/métricas), **agentes** (planificación), o **bots** de apoyo (QA/soporte técnico)?
3) ¿Tienes un caso específico? (p. ej., predicción de fallos en motores, trayectoria orbital,
fusión de sensores, control/guías, mantenimiento predictivo)  Dime eso y te propongo un guion
breve + un mini ejemplo con LangChain.

Usuario: Explica LCEL en una frase.
Asistente:
LCEL (LangChain Expression Language) es una forma declarativa de encadenar componentes de
LangChain para construir pipelines (p. ej., prompt → modelo → parsers) con una sintaxis

## 7. RAG con LangChain y el PDF del BOE

LangChain proporciona cargadores de documentos, splitters, embeddings, vector stores y retrievers. El siguiente ejemplo reproduce un RAG compacto con `BOE-A-2015-2870.pdf`.

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = Path("BOE-A-2015-2870.pdf")

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=180,
)
splits = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore.from_documents(splits, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Paginas cargadas:", len(docs))
print("Fragmentos creados:", len(splits))

Paginas cargadas: 7
Fragmentos creados: 35


Construimos una cadena RAG explicita. Primero recupera documentos; despues formatea el contexto; finalmente llama al modelo.

In [11]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(documents):
    blocks = []
    for i, doc in enumerate(documents, start=1):
        page = doc.metadata.get("page", 0) + 1
        blocks.append(f"[Fuente {i} | pag. {page}]\n{doc.page_content}")
    return "\n\n".join(blocks)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente docente. Responde solo con el contexto proporcionado y cita paginas."),
    ("human", "Pregunta: {question}\n\nContexto:\n{context}\n\nRespuesta:"),
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = "Cual es el objeto de la Orden IET/457/2015?"
answer = rag_chain.invoke(question)
show(answer)

El objeto de la **Orden IET/457/2015** es **modificar la Orden IET/786/2013, de 7 de mayo**,
por la que se establecen las **bases reguladoras** para la concesión de **ayudas en el ámbito
de las tecnologías de la información y las comunicaciones (TIC) y la sociedad de la
información**, dentro del **Plan de Investigación Científica y Técnica y de Innovación
2013-2016**, en el marco de la **acción estratégica de economía y sociedad digital**. (Fuente
1, p. 1)


### Inspeccionar recuperacion

La respuesta final solo es una parte del sistema. Conviene mirar que fragmentos fueron recuperados antes de confiar en el resultado.

In [12]:
retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    page = doc.metadata.get("page", 0) + 1
    print(f"\nFuente {i} | pagina {page}")
    show(doc.page_content[:600])


Fuente 1 | pagina 1
BOLETÍN OFICIAL DEL ESTADO Núm. 65 Martes 17 de marzo de 2015 Sec. III.   Pág. 24131 III. OTRAS
DISPOSICIONES MINISTERIO DE INDUSTRIA, ENERGÍA Y TURISMO 2870 Orden IET/457/2015, de 11 de
marzo, por la que se modifica la Orden  IET/786/2013, de 7 de mayo, por la que se establecen
las bases reguladoras  de la concesión de ayudas en el ámbito de las tecnologías de la
información y  las comunicaciones (TIC) y la sociedad de la información, dentro del Plan de
Investigación Científica y Técnica y de Innovación 2013-2016 en el marco de  la acción
estratégica de economía y sociedad digital. La Orden

Fuente 2 | pagina 1
del Consejo y el Reglamento (UE) n.º 1301/2013 del Parlamento Europeo y del Consejo  de 17 de
diciembre de 2013 sobre el Fondo Europeo de Desarrollo Regional y sobre  disposiciones
específicas relativas al objetivo de inversión en crecimiento y empleo y por  el que se deroga
el Reglamento (CE) n.º 1080/2006. Debido a estos cambios, es preciso proceder a la 

## 8. Guardrail simple para RAG

En documentos normativos, la cadena debe evitar contestar si el contexto recuperado no es suficiente. Este prompt fuerza una respuesta de no cobertura cuando la pregunta no aparece en el documento.

In [13]:
guarded_rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Responde solo con el contexto. Si el contexto no contiene la respuesta, di exactamente: "
        "'No aparece en los fragmentos recuperados'. No inventes datos.",
    ),
    ("human", "Pregunta: {question}\n\nContexto:\n{context}\n\nRespuesta:"),
])

guarded_rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | guarded_rag_prompt
    | llm
    | StrOutputParser()
)

show(guarded_rag_chain.invoke("Que avion comercial tiene mayor alcance en 2026?"))

No aparece en los fragmentos recuperados


## 9. Cierre

Ideas principales:

- LangChain organiza aplicaciones LLM como componentes invocables.
- LCEL permite componer prompts, modelos, parsers, retrievers y funciones con `|`.
- La salida estructurada reduce ambiguedad cuando la respuesta alimenta codigo.
- Las herramientas deben ejecutar codigo controlado por la aplicacion, no instrucciones arbitrarias del modelo.
- En RAG, inspeccionar la recuperacion es tan importante como leer la respuesta final.
- Para produccion conviene persistir indices, medir calidad de recuperacion y anadir evaluaciones.